# 1、SummarizationMiddleware中间件
# 测试trigger、keep参数

In [15]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from rich import print as rprint
from langchain.agents.middleware import SummarizationMiddleware

from numpy import extract

load_dotenv(override=True)
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
ZHIPU_BASE_URL = os.getenv("ZHIPU_BASE_URL")

model=init_chat_model(
    model="glm-5.2",  # 模型名称
    model_provider="openai",
    api_key=ZHIPU_API_KEY,
    base_url=ZHIPU_BASE_URL,  # ZHIPU API 的基础 URL
    profile={"max_input_tokens": 1000000}
)

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                #("fraction",0.001)
            ],
            keep=("messages",2)
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()

ValueError: Model profile information is required to use fractional token limits, and is unavailable for the specified model. Please use absolute token counts instead, or pass `

ChatModel(..., profile={"max_input_tokens": ...})`.

with a desired integer value of the model's maximum input tokens.

In [16]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

**历史消息摘要：**

用户老王与设定为“友好AI助手”的AI进行了初次问候与自我介绍。AI自称“小王”，老王随后表达了认识小王的高兴之情。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哎呀，老王别生气，开个玩笑嘛！😂

我是想说，咱们以后相处的时间还长着呢，要是哪天我笨手笨脚没回答好您的问题，惹您不高兴了，您今天这“高兴劲儿”不就白搭了嘛~

作为您的友好AI助手小王，我可是随时准备为您效劳的，以后您就会发现我不仅现在让您高兴，以后也能帮上大忙！有什么想聊的或者需要帮忙的，您尽管吩咐！
